# Training and Evaluating a Feed Forward Model on GIFT-Eval


This notebook demonstrates how to train and evaluate a GluonTS [Simple Feed Forward model](https://ts.gluon.ai/dev/api/gluonts/gluonts.torch.model.simple_feedforward.html) on the GIFT-Eval benchmark.


## Dataset


Define the datasets we'll use. For the sake of brevity, we'll only use two datasets.


In [1]:
import json
from dotenv import load_dotenv
from datetime import datetime

format = "%m/%d/%Y %I:%M:%S%p"


def print_timestamp():
    now = datetime.now()
    formatted_time = now.strftime(format)
    print(f"Last time ran: {formatted_time}")


# Load environment variables
load_dotenv()

# Create a set of all the short dataset names
short_dataset_names_string = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
short_datasets_names = set(short_dataset_names_string.split())

# Name of the short dataset we'll use
short_dataset = "m4_hourly"

# Create a set of all the medium to long dataset names
med_long_dataset_names_string = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
med_long_datasets_names = set(med_long_dataset_names_string.split())

# Name of the medium to long dataset we'll use
med_long_dataset = "bizitobs_l2c/H"

# Combine all datasets names into one list
all_dataset_names = [short_dataset, med_long_dataset]

# Load the dataset properties map
dataset_properties_map = json.load(open("dataset_properties.json"))

print_timestamp()

Last time ran: 03/05/2025 10:44:07PM


Combine all the datasets into one GluonTS `ListDataset`


In [2]:
from gift_eval.data import Dataset
from gluonts.dataset.common import ListDataset


def check_if_multivariate(dataset_name, term):
    # Get the dataset's target dimension
    target_dimension = Dataset(
        name=dataset_name,
        term=term,
        to_univariate=False,
    ).target_dim

    # Check if the dataset is already univariate
    return target_dimension > 1


terms = ["short", "medium", "long"]


def combine_datasets():
    combined_train_data = []
    combined_val_data = []

    for dataset_name in all_dataset_names:
        for term in terms:
            term_is_not_short = term != "short"
            dataset_is_short = dataset_name not in med_long_datasets_names

            if term_is_not_short and dataset_is_short:
                continue

            # Check if the dataset is multivariate
            is_multivariate = check_if_multivariate(dataset_name, term)

            # True if the dataset is multivariate
            to_univariate = True if is_multivariate else False

            # Initialize dataset
            dataset = Dataset(name=dataset_name, term=term, to_univariate=to_univariate)

            # Add train and val data
            combined_train_data.extend(dataset.training_dataset)
            combined_val_data.extend(dataset.validation_dataset)

    return combined_train_data, combined_val_data


combined_train_data, combined_val_data = combine_datasets()

# Combine all datasets into one
train_data = ListDataset(combined_train_data, freq="1H")
val_data = ListDataset(combined_val_data, freq="1H")

print_timestamp()

Last time ran: 03/05/2025 10:44:08PM


## Training


Initialize a feed forward neural network


In [3]:
from gluonts.torch.model.simple_feedforward import SimpleFeedForwardEstimator

# Define hyperparameters
trainer_kwargs = {"max_epochs": 1}

# Instantiate feed forward neural network
estimator = SimpleFeedForwardEstimator(
    prediction_length=48,
    context_length=48,
    trainer_kwargs=trainer_kwargs,
)

print_timestamp()

Last time ran: 03/05/2025 10:44:11PM


Train the model on the combined dataset to get a GluonTS `Predictor`


In [4]:
print_timestamp()

# Define hyperparameters
trainer_kwargs = {"max_epochs": 1}

predictor = estimator.train(train_data, val_data)

/home/mike_gee/miniconda3/envs/tempo/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mike_gee/miniconda3/envs/tempo/lib/python3.11/ ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name  | Type                   | Params | Mode 
---------------------------------------------------------
0 | model | SimpleFeedForwardModel | 21.2 K | train
---------------------------------------------------------
21.2 K    Trainable params
0         Non-trainable params
21.2 K    Total params
0.085     Total estimated model params size (MB)
11        Modules in train mode
0         Modules in eval mode


Last time ran: 03/05/2025 10:44:11PM


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 0, global step 50: 'val_loss' reached 5.96067 (best 5.96067), saving model to '/home/mike_gee/TEMPO/gift-eval/notebooks/lightning_logs/version_30/checkpoints/epoch=0-step=50.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=1` reached.


## Inference


Initialize a CSV file to save our model's performance on each dataset


In [5]:
import csv
import os

# Name of the directory where our model's results will be saved
output_directory = "results"

# Ensure the directory exists
os.makedirs(output_directory, exist_ok=True)

# Define the CSV file's path
csv_file_path = os.path.join(output_directory, "feedforward.csv")

# Initialize the CSV file's header
with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)

    # Write the header
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
        ]
    )

print_timestamp()

Last time ran: 03/05/2025 10:44:11PM


Define some helper functions for evaluation


In [6]:
def get_dataset_key(dataset_name):
    return dataset_name.split("/")[0] if "/" in dataset_name else dataset_name.lower()


def get_dataset_freq(dataset_name):
    key = get_dataset_key(dataset_name)
    return (
        dataset_name.split("/")[1]
        if "/" in dataset_name
        else dataset_properties_map[key]["frequency"]
    )


def get_dataset_config(dataset_name, term):
    key = get_dataset_key(dataset_name)
    freq = get_dataset_freq(dataset_name)
    return f"{key}/{freq}/{term}"


def write_to_csv(results, csv_file_path, term, dataset_name):
    # Initialize dataset's configuation
    dataset_config = get_dataset_config(dataset_name, term)

    # Write the results to the CSV file
    with open(csv_file_path, "a", newline="") as csvfile:
        key = get_dataset_key(dataset_name)

        writer = csv.writer(csvfile)
        writer.writerow(
            [
                dataset_config,
                "feedforward",
                results["MSE[mean]"][0],
                results["MSE[0.5]"][0],
                results["MAE[0.5]"][0],
                results["MASE[0.5]"][0],
                results["MAPE[0.5]"][0],
                results["sMAPE[0.5]"][0],
                results["MSIS"][0],
                results["RMSE[mean]"][0],
                results["NRMSE[mean]"][0],
                results["ND[0.5]"][0],
                results["mean_weighted_sum_quantile_loss"][0],
                dataset_properties_map[key]["domain"],
                dataset_properties_map[key]["num_variates"],
            ]
        )

Load the metrics we'll evaluate our model on


In [7]:
from gluonts.ev.metrics import (
    MSE,
    MAE,
    MASE,
    MAPE,
    SMAPE,
    MSIS,
    RMSE,
    NRMSE,
    ND,
    MeanWeightedSumQuantileLoss,
)

# Instantiate metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

print_timestamp()

Last time ran: 03/05/2025 10:44:12PM


Perform inference on each dataset we trained our model on


In [8]:
from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality


def perform_inference(predictor, term, dataset_name):
    # Check if the dataset is multivariate
    is_multivariate = check_if_multivariate(dataset_name, term)

    # True if the dataset is multivariate
    to_univariate = True if is_multivariate else False

    # Initialize dataset
    dataset = Dataset(name=dataset_name, term=term, to_univariate=to_univariate)

    # Get the seasonal component's length
    season_length = get_seasonality(dataset.freq)

    # Evaluate the model on the test set
    results = evaluate_model(
        predictor,
        test_data=dataset.test_data,
        metrics=metrics,
        batch_size=512,
        axis=None,
        mask_invalid_label=True,
        allow_nan_forecast=False,
        seasonality=season_length,
    )

    return results


for term in terms:
    for dataset_name in all_dataset_names:
        # Only perform inference on datasets where prediction length is 48
        if term != "short":
            continue

        print(f"Performing inference on {dataset_name}")

        # Get the results from performing inference
        results = perform_inference(predictor, term, dataset_name)

        # Write the results to the csv file
        write_to_csv(results, csv_file_path, term, dataset_name)
        print(f"Wrote results for {dataset_name} to {csv_file_path}")

print_timestamp()

Performing inference on m4_hourly


414it [00:09, 45.29it/s]


Wrote results for m4_hourly to results/feedforward.csv
Performing inference on bizitobs_l2c/H


42it [00:00, 45.91it/s]

Wrote results for bizitobs_l2c/H to results/feedforward.csv
Last time ran: 03/05/2025 10:44:22PM


Display the results obtained from inference as a pandas `DataFrame`


In [9]:
import pandas as pd

print_timestamp()
df = pd.read_csv("./results/feedforward.csv")
df.head()

Last time ran: 03/05/2025 10:44:22PM


,dataset,model,eval_metrics/MSE[mean],eval_metrics/MSE[0.5],eval_metrics/MAE[0.5],eval_metrics/MASE[0.5],eval_metrics/MAPE[0.5],eval_metrics/sMAPE[0.5],eval_metrics/MSIS,eval_metrics/RMSE[mean],eval_metrics/NRMSE[mean],eval_metrics/ND[0.5],eval_metrics/mean_weighted_sum_quantile_loss,domain,num_variates
0,m4_hourly/H/short,feedforward,3.525784e+07,3.525784e+07,1019.245572,10.138773,0.777589,0.326749,235.283299,5937.831282,0.810645,0.139150,0.170109,Econ/Fin,1
1,bizitobs_l2c/H/short,feedforward,3.134752e+02,3.134752e+02,14.196777,1.356354,1.404702,1.090114,17.951358,17.705232,0.954357,0.765243,0.641458,Web/CloudOps,7
